# EDA

In [178]:
%load_ext autoreload
%autoreload 2

import os
from datetime import timedelta

os.chdir("..")  # repo root, so `src` imports

from src.data import loader
from src.plotting.charts import calendar_plot, carpet_plot, grid, line_chart
from src.plotting.display import show_df
from plotly.subplots import make_subplots

from datetime import date

import polars as pl
from scipy.stats import ks_2samp

from src.plotting.charts import acf_chart, heatmap, line_chart, histogram
from src.plotting.display import show_df
from src.stats.distribution_shift import psi, rolling_moments, spike_flag
from src.stats.independence import acf_by_group, acf_frame, ljung_box
from src.stats.stationarity import adf_kpss

from src.constants import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load each source

### OMIE
`marginalpdbc`: day-ahead marginal price
<br>
`curva_pbc`: aggregate bid curves.

In [179]:
omie_price = loader.load("omie", "marginalpdbc")

show_df(
    omie_price.drop("source_file").tail(10)
)
show_df(
    omie_price.drop("source_file")
    .describe()
)

,year,month,day,period,price_pt,price_es,date,datetime
0,"2,026",9,27,87,183.76,183.76,2026-09-27,2026-09-27 21:30
1,"2,026",9,27,88,180.00,180.00,2026-09-27,2026-09-27 21:45
2,"2,026",9,27,89,183.00,183.00,2026-09-27,2026-09-27 22:00
3,"2,026",9,27,90,178.45,178.45,2026-09-27,2026-09-27 22:15
4,"2,026",9,27,91,172.32,172.32,2026-09-27,2026-09-27 22:30
5,"2,026",9,27,92,167.14,167.14,2026-09-27,2026-09-27 22:45
6,"2,026",9,27,93,168.36,168.36,2026-09-27,2026-09-27 23:00
7,"2,026",9,27,94,166.78,166.78,2026-09-27,2026-09-27 23:15
8,"2,026",9,27,95,165.20,165.20,2026-09-27,2026-09-27 23:30
9,"2,026",9,27,96,160.00,160.00,2026-09-27,2026-09-27 23:45


,statistic,year,month,day,period,price_pt,price_es,date,datetime
0,count,"93,671.00","93,671.00","93,671.00","93,671.00","93,671.00","93,671.00",93671,93671
1,null_count,0.00,0.00,0.00,0.00,0.00,0.00,0,0
2,mean,"2,023.31",6.41,15.66,25.78,79.28,78.97,2023-10-20 21:56:19.474971,2023-10-21 09:34:37.501041
3,std,2.44,3.42,8.77,24.81,63.75,63.98,,
4,min,"2,019.00",1.00,1.00,1.00,-10.00,-15.00,2019-01-01,2019-01-01 00:00:00
5,25%,"2,021.00",3.00,8.00,9.00,33.41,33.14,2021-09-03,2021-09-03 19:00:00
6,50%,"2,024.00",6.00,16.00,17.00,67.37,66.58,2024-05-07,2024-05-07 12:00:00
7,75%,"2,026.00",9.00,23.00,31.00,112.64,112.64,2026-01-27,2026-01-27 00:30:00
8,max,"2,026.00",12.00,31.00,100.00,651.00,700.00,2026-09-27,2026-09-27 23:45:00


In [180]:
def plot_grid(
    figs: list,
    rows: int,
    cols: int,
    width: int = 1400,
    height: int = 450,
):
    out = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f.layout.title.text or "" for f in figs],
    )
    for i, f in enumerate(figs):
        out.add_traces(list(f.data), rows=i // cols + 1, cols=i % cols + 1)
    return out.update_layout(width=width, height=height, template="plotly_white", showlegend=False)

plot_grid(
    [
        line_chart(omie_price, "datetime", "price_es", every="1d", title="Day-ahead price ES (daily mean)"),
        carpet_plot(omie_price, "datetime", "price_es", title="Price ES: hour of day x day"),
    ],
    rows=1,
    cols=2,
)

Comments: 
- Price spiked in 2022, probably due to the conflict in Ukraine
- After 2022, energy seems to be cheaper from 1pm to 6pm. Probably due to a higher share of renewables like solar which produces more during the day. 

In [181]:
omie_price_min_date = omie_price.select(pl.col("date").min()).item()
omie_price_max_date = omie_price.select(pl.col("date").max()).item()

print(
    f"Number of records per day (first day): {omie_price.filter(pl.col('date') == omie_price_min_date).height}\n"
    f"Number of records per day (last day): {omie_price.filter(pl.col('date') == omie_price_max_date).height}"
)

Number of records per day (first day): 24
Number of records per day (last day): 96


In [182]:
switch = omie_price.filter(
    pl.col("datetime").shift(-1) - pl.col("datetime") == pl.duration(minutes=15)
)["datetime"].first()
print(
    f"Break happened at {switch}"
)

Break happened at 2025-10-01 00:00:00


- Before 2025, we had 24 daily records (one per hour).
- After 2025-10-01, contracts were every 15min (24*4=96). 

Let's compute rolling mean and std.

In [183]:
omie_price

year,month,day,period,price_pt,price_es,source_file,date,datetime
i32,i32,i32,i32,f64,f64,str,date,datetime[μs]
2019,1,1,1,66.88,66.88,"""marginalpdbc_20190101.1""",2019-01-01,2019-01-01 00:00:00
2019,1,1,2,66.88,66.88,"""marginalpdbc_20190101.1""",2019-01-01,2019-01-01 01:00:00
2019,1,1,3,66.0,66.0,"""marginalpdbc_20190101.1""",2019-01-01,2019-01-01 02:00:00
2019,1,1,4,63.64,63.64,"""marginalpdbc_20190101.1""",2019-01-01,2019-01-01 03:00:00
2019,1,1,5,58.85,58.85,"""marginalpdbc_20190101.1""",2019-01-01,2019-01-01 04:00:00
…,…,…,…,…,…,…,…,…
2026,9,27,92,167.14,167.14,"""marginalpdbc_20260927.1""",2026-09-27,2026-09-27 22:45:00
2026,9,27,93,168.36,168.36,"""marginalpdbc_20260927.1""",2026-09-27,2026-09-27 23:00:00
2026,9,27,94,166.78,166.78,"""marginalpdbc_20260927.1""",2026-09-27,2026-09-27 23:15:00


In [184]:
WINDOWS = [30, 90]  # days
REGIMES = {
    # "pre-crisis": date(2019, 1, 1),
    # "gas crisis": date(2021, 7, 1),
    # "iberian cap": date(2022, 6, 15),
    # "post-cap": date(2024, 1, 1),
    "1h": omie_price_min_date,
    "15-min": date(2025, 10, 1),
}

regimes = pl.DataFrame({"regime": list(REGIMES), "date": list(REGIMES.values())})

# Add regimes
stats = omie_price.sort("datetime").join_asof(
    regimes.with_columns(pl.col("date").cast(omie_price.schema["datetime"])),
    left_on="datetime",
    right_on="date",
)

for w in WINDOWS:
    roll = dict(by="datetime", window_size=f"{w}d")
    stats = stats.with_columns(
        mean=pl.col(PRICE_ES).rolling_mean_by(**roll).over("regime"),
        std=pl.col(PRICE_ES).rolling_std_by(**roll).over("regime"),
    ).with_columns(
        resid=pl.col(PRICE_ES) - pl.col("mean"),
    ).with_columns(
        rmse=(pl.col("resid") ** 2).rolling_mean_by(**roll).sqrt().over("regime"), # Neg values
        resid_var=pl.col("resid").rolling_var_by(**roll).over("regime"),
    ).rename(lambda c: f"{c}_{w}d" if c in ("mean", "std", "resid", "rmse", "resid_var") else c)

# Added features
new_features = set(stats.columns) - set(omie_price.columns)
new_features.remove("date_right")

for part in stats.select(new_features).partition_by("regime"):
    show_df(
        part.describe(),
        caption=f"Stats for regime {part['regime'][0]}",
    )

,statistic,rmse_90d,resid_90d,resid_var_90d,resid_30d,regime,rmse_30d,mean_90d,resid_var_30d,std_90d,std_30d,mean_30d
0,count,"59,111.00","59,111.00","59,110.00","59,111.00",59111,"59,111.00","59,111.00","59,110.00","59,110.00","59,110.00","59,111.00"
1,null_count,0.00,0.00,1.00,0.00,0,0.00,0.00,1.00,1.00,1.00,0.00
2,mean,34.47,0.05,"1,336.94",0.01,,30.38,82.73,"1,155.53",31.53,28.27,82.77
3,std,20.40,40.47,"1,422.88",36.56,,20.06,53.54,"1,537.27",19.11,18.49,55.72
4,min,0.00,-234.41,0.00,-249.39,1h,0.00,18.42,0.00,0.00,0.00,5.08
5,25%,11.60,-17.60,114.00,-13.93,,10.79,44.29,104.24,10.52,10.23,42.98
6,50%,38.22,1.22,"1,064.05",1.65,,30.01,68.61,818.39,33.54,28.20,66.31
7,75%,48.03,19.22,"1,922.63",15.81,,43.00,97.48,"1,666.80",44.83,41.64,104.50
8,max,86.93,471.28,"7,048.46",458.18,1h,113.34,235.96,"12,202.64",84.87,108.60,287.86


,statistic,rmse_90d,resid_90d,resid_var_90d,resid_30d,regime,rmse_30d,mean_90d,resid_var_30d,std_90d,std_30d,mean_30d
0,count,"34,560.00","34,560.00","34,559.00","34,560.00",34560,"34,560.00","34,560.00","34,559.00","34,559.00","34,559.00","34,560.00"
1,null_count,0.00,0.00,1.00,0.00,0,0.00,0.00,1.00,1.00,1.00,0.00
2,mean,49.28,6.79,"2,286.73",2.44,,49.48,65.67,"2,466.47",47.55,47.24,70.02
3,std,11.18,56.36,930.51,53.45,,13.79,20.10,"1,542.77",9.92,14.68,30.06
4,min,0.00,-120.13,0.09,-137.02,15-min,0.00,30.89,0.09,0.61,0.61,15.70
5,25%,42.58,-42.51,"1,723.70",-39.81,,41.98,49.91,"1,545.92",41.85,37.80,52.12
6,50%,46.80,6.13,"2,022.51",4.57,,47.26,67.05,"2,020.92",44.80,45.27,67.73
7,75%,49.99,49.90,"2,314.72",44.79,,52.71,74.18,"2,685.31",48.51,52.19,83.94
8,max,81.16,218.20,"5,490.21",217.08,15-min,84.87,120.14,"7,120.32",75.25,84.61,137.27


- Prices appear to be cheaper (from 83 30D to 70 30D)
- Prices are more volatile. 30D std went from 28 on avg to 47. Almost double. Good for PnL -> more variance. 
- RMSE > std so prices are drifting. Less important for 15min. Little trend + noise. 
- Std / Mean: around 34% for hourly and 65% for 15min. Bigger spread available.

Stationarity analysis

In [185]:
daily = stats.group_by(pl.col("datetime").dt.date().alias("date"), "regime").agg(pl.col(PRICE_ES).mean()).sort("date")


transforms = {
    "level": pl.col(PRICE_ES),
    "asinh": pl.col(PRICE_ES).arcsinh(), # Used for stabilizing variance (normal distribution), use due to negative values instead of logs
    "diff_1d": pl.col(PRICE_ES).diff(1),
    "diff_7d": pl.col(PRICE_ES).diff(7),
}

parts = {"all": daily} | {k[0]: v for k, v in daily.partition_by("regime", as_dict=True, maintain_order=True).items()}

show_df(
    pl.DataFrame(
        [
            {"regime": regime, "transform": name, **adf_kpss(part.select(expr).to_series())}

            # ADF: H0: non-stationary, H1: stationary
            # KPSS: H0: stationary, H1: non-stationary

            for regime, part in parts.items()
            for name, expr in transforms.items()
        ],
    ),
    precision=3,
)

# I need ADF p < 0.05 and KPSS p > 0.05 to conclude stationarity.

first_weeks = daily.filter(pl.col("date") < omie_price_min_date + timedelta(days=7 * 5))

fig = line_chart(
    first_weeks,
    "date",
    PRICE_ES,
    title="Daily mean prices (first five weeks)",
)
fig.add_scatter(
    x=first_weeks["date"],
    y=first_weeks[PRICE_ES].arcsinh(),
    mode="lines",
    name="asinh transform",
    yaxis="y2",
)
fig.add_scatter(
    x=first_weeks["date"],
    y=first_weeks[PRICE_ES].diff(1),
    mode="lines",
    name="1-day difference",
    yaxis="y2",
)

,regime,transform,adf_stat,adf_p,kpss_stat,kpss_p
0,all,level,-2.547,0.104,0.994,0.010
1,all,asinh,-3.697,0.004,0.691,0.014
2,all,diff_1d,-13.343,0.000,0.056,0.100
3,all,diff_7d,-11.290,0.000,0.030,0.100
4,1h,level,-2.183,0.212,1.224,0.010
5,1h,asinh,-3.105,0.026,0.919,0.010
6,1h,diff_1d,-12.367,0.000,0.065,0.100
7,1h,diff_7d,-13.259,0.000,0.030,0.100
8,15-min,level,-1.307,0.626,1.188,0.010
9,15-min,asinh,-2.719,0.071,0.764,0.010


- Raw levels are obviously not stationary. 
- ADF is conclusive but kpss rejects stationarity. Looking at the chart, price seems rough but transformations seem to smooth. A little trend is probably causing the test to fail. 
- 1D differeencing is the most convincing with ADF and KPSS concluding stationarity.

Drift

In [186]:
# PSI (Population Stability Index)
# Measures stability over time.
# PSI > 0.25 is a shift

rows = []
for regime, part in [
    ("1h", omie_price.filter(pl.col("datetime") < switch)),
    ("15-min", omie_price.filter(pl.col("datetime") >= switch)),
]:
    years = part["year"].unique().sort().to_list()
    for prev, y in zip(years, years[1:]):
        a = part.filter(pl.col("year") == prev)[PRICE_ES]
        b = part.filter(pl.col("year") == y)[PRICE_ES]
        rows.append({"regime": regime, "year": y, "psi": psi(a, b), "ks_stat": ks_2samp(a, b).statistic})

show_df(pl.DataFrame(rows))

,regime,year,psi,ks_stat
0,1h,"2,020",1.58,0.51
1,1h,"2,021",3.05,0.78
2,1h,"2,022",1.17,0.47
3,1h,"2,023",3.65,0.59
4,1h,"2,024",0.31,0.25
5,1h,"2,025",0.05,0.06
6,15-min,"2,026",0.71,0.21


- 5/7 years show shift. The data used to have a strong trend (probably Ukraine conflict). 

In [187]:
hourly = omie_price.sort("datetime").group_by_dynamic("datetime", every="1h").agg(pl.col(PRICE_ES).mean()).with_columns(year=pl.col("datetime").dt.year())

# 1D differencing
hourly = hourly.with_columns(pl.col(PRICE_ES).diff(24).alias(PRICE_ES_DIFF_1D))

show_df(
    hourly.group_by("year")
    .agg(
        pl.col(PRICE_ES_DIFF_1D).skew().alias("skew"),
        pl.col(PRICE_ES_DIFF_1D).kurtosis().alias("kurt"),
        pl.col(PRICE_ES_DIFF_1D).quantile(0.01).alias("q01"),
        pl.col(PRICE_ES_DIFF_1D).median().alias("q50"),
        pl.col(PRICE_ES_DIFF_1D).quantile(0.99).alias("q99"),
        (pl.col(PRICE_ES_DIFF_1D) <= 0).mean().alias("positive_share"),
    )
    .sort("year"),
)

histogram(omie_price, PRICE_ES, nbins=100, title="Distribution of day-ahead price ES")


,year,skew,kurt,q01,q50,q99,positive_share
0,"2,019",0.23,3.60,-21.97,-0.10,22.87,0.52
1,"2,020",0.20,3.02,-19.15,0.01,21.26,0.50
2,"2,021",-0.62,13.69,-88.60,0.60,77.35,0.48
3,"2,022",-0.04,3.85,-111.56,-1.25,111.66,0.52
4,"2,023",0.46,3.36,-85.23,-0.75,98.36,0.53
5,"2,024",0.29,2.77,-77.00,0.00,81.70,0.53
6,"2,025",0.28,1.90,-73.94,-0.20,87.58,0.52
7,"2,026",0.21,3.22,-78.23,0.00,84.36,0.50


- Distribution is right skewed, with frequent fat tails. Positive jumps are larger than drops but median is 0 so changes are symmetric.

In [199]:
# Roughness ratio
# Roughness ratio = rolling std of 1D diff / rolling std of price

ROUGH_WINDOW = 24 * 7  # hours

rough = hourly.with_columns(
    (
        pl.col(PRICE_ES).diff().rolling_std(ROUGH_WINDOW)
        / pl.col(PRICE_ES).rolling_std(ROUGH_WINDOW)
    ).alias("roughness_ratio"),
)

r = pl.col("roughness_ratio")

show_df(
    rough.group_by("year")
    .agg(
        r.count().alias("count"),
        r.null_count().alias("null_count"),
        r.mean().alias("mean"),
        r.std().alias("std"),
        r.min().alias("min"),
        r.quantile(0.25).alias("25%"),
        r.median().alias("50%"),
        r.quantile(0.75).alias("75%"),
        r.max().alias("max"),
    )
    .sort("year")
    .with_columns(pl.col("year").cast(pl.String))
    .transpose(include_header=True, header_name="statistic", column_names="year"),
    caption="Roughness ratio by year",
)


carpet_plot(rough, "datetime", "roughness_ratio", size="small", title=f"Roughness ratio ({ROUGH_WINDOW}h window)")

rho = (1 - pl.col("roughness_ratio") ** 2 / 2).mean()
print(f"Average correlation with the previous hour: {rough.select(rho).item():.2f}")

,statistic,2019,2020,2021,2022,2023,2024,2025,2026
0,count,"8,591.00","8,759.00","8,759.00","8,759.00","8,735.00","8,783.00","8,711.00","6,479.00"
1,null_count,168.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,mean,0.42,0.40,0.40,0.46,0.41,0.44,0.45,0.43
3,std,0.08,0.07,0.09,0.07,0.07,0.12,0.09,0.08
4,min,0.20,0.22,0.15,0.27,0.17,0.20,0.20,0.25
5,25%,0.37,0.35,0.34,0.41,0.36,0.37,0.39,0.38
6,50%,0.42,0.39,0.40,0.47,0.41,0.42,0.43,0.41
7,75%,0.48,0.44,0.47,0.52,0.44,0.49,0.50,0.45
8,max,0.65,0.62,0.61,0.67,0.64,1.04,0.75,0.82


Average correlation with the previous hour: 0.91


Intuition:

- The average is 60 €/MWh and most hours sit between about 40 and 80, so std(price) ≈ 20.
- From one hour to the next, the price usually moves by around ±8, so std(Δprice) ≈ 8.
- Roughness = 8 / 20 = 0.4.
<br>

Interpretation:
- Price step is around 40% of the weekly price standard deviation.<br>
- Prices seem very persistent with a correlation of 0.9 with the previous hour. 

Autocorrelation

In [206]:
# Let's expand to more lags
plots = []

# ACF
ACF_LAGS = 24 * 8

plots.append(
    acf_chart(
        acf_frame(hourly[PRICE_ES], ACF_LAGS),
        size="small",
        title="ACF, hourly price"
    )
)
plots.append(
    acf_chart(
        acf_frame(hourly[PRICE_ES_DIFF_1D], ACF_LAGS),
        size="small",
        title="ACF, hourly price (1D diff)"
    )
)

# PACF
# More specific information about the lags, but more sensitive to noise.

plots.append(acf_chart(
        acf_frame(hourly[PRICE_ES], ACF_LAGS, partial=True),
        size="small",
        title="PACF, hourly price"
    )
)
plots.append(acf_chart(
        acf_frame(hourly[PRICE_ES_DIFF_1D], ACF_LAGS, partial=True),
        size="small",
        title="PACF, hourly price (1D diff)"
    )
)

plot_grid(
    plots,
    rows=2,
    cols=2,
    width=1400,
    height=450,
)

- Strong memory which slowly decays. Autocorrelation remains above 0.6 after 192 hours, consistent with non-stationarity (as shown by ADF/KPSS).
- On 1-level differencing, it breaks at lag 18.
- Lag 1 shows the strongest correlation (0.97), consistent with the high persistence found with the roughness ratio.
- I would keep lags 1 and 2 (strongly negative), and lags 20 to 25 as a group (same hour yesterday). I would also test lags 48 and 168 (same hour 2 days and 1 week ago), and keep the final set based on validation error.

In [213]:
# what calendar features can't explain

hourly = hourly.with_columns(
    detrended=pl.col(PRICE_ES) - pl.col(PRICE_ES).rolling_mean(24 * 30),
).with_columns(
    resid=pl.col("detrended") - pl.col("detrended").mean().over("hour", "weekday"),
)

show_df(ljung_box(hourly["resid"], [24, 48, 168]), precision=1)

,lag,lb_stat,lb_pvalue
0,24,"557,869.8",0.0
1,48,"867,806.9",0.0
2,168,"1,447,923.6",0.0


- After removing the average price for each hour × weekday, the residuals are still strongly autocorrelated at 24, 48 and 168 hours (p = 0 everywhere)

In [ ]:
omie_price = loader.load("omie", "marginalpdbc")

_, last = loader.available_days(loader.SOURCE_DIRS["omie"], "curva_pbc")
omie_curves = loader.load("omie", "curva_pbc", last - timedelta(days=6), last)

print(omie_price.shape, omie_curves.shape)
omie_price.tail()

(93671, 8) (3698458, 10)


year,month,day,period,price_pt,price_es,source_file,date
i32,i32,i32,i32,f64,f64,str,date
2026,9,27,92,167.14,167.14,"""marginalpdbc_20260927.1""",2026-09-27
2026,9,27,93,168.36,168.36,"""marginalpdbc_20260927.1""",2026-09-27
2026,9,27,94,166.78,166.78,"""marginalpdbc_20260927.1""",2026-09-27
2026,9,27,95,165.2,165.2,"""marginalpdbc_20260927.1""",2026-09-27
2026,9,27,96,160.0,160.0,"""marginalpdbc_20260927.1""",2026-09-27


### ESIOS

In [ ]:
esios = {v: loader.load("esios", v) for v in loader.variables("esios")}

for name, df in esios.items():
    print(f"{name:28} {df.shape}  {df['datetime_utc'].min()} -> {df['datetime_utc'].max()}")

demand_forecast              (182425, 4)  2018-12-31 23:00:00+00:00 -> 2026-09-30 21:45:00+00:00
demand_forecast_d1           (67847, 4)  2018-12-31 23:00:00+00:00 -> 2026-09-27 21:00:00+00:00
imbalance_net_volume         (115436, 4)  2018-12-31 23:00:00+00:00 -> 2026-09-26 21:15:00+00:00
imbalance_price_down         (115693, 4)  2018-12-31 23:00:00+00:00 -> 2026-09-26 21:15:00+00:00
imbalance_price_up           (115693, 4)  2018-12-31 23:00:00+00:00 -> 2026-09-26 21:15:00+00:00
residual_demand_forecast     (182431, 4)  2018-12-31 23:00:00+00:00 -> 2026-09-30 21:45:00+00:00
solar_pv_forecast            (182407, 4)  2018-12-31 23:00:00+00:00 -> 2026-09-30 21:45:00+00:00
solar_pv_forecast_d1         (67847, 4)  2018-12-31 23:00:00+00:00 -> 2026-09-27 21:00:00+00:00
wind_forecast                (182351, 4)  2018-12-31 23:00:00+00:00 -> 2026-09-30 21:45:00+00:00
wind_forecast_d1             (67847, 4)  2018-12-31 23:00:00+00:00 -> 2026-09-27 21:00:00+00:00


### ENTSO-E

In [ ]:
entsoe = {v: loader.load("entsoe", v) for v in loader.variables("entsoe")}

for name, df in entsoe.items():
    print(f"{name:28} {df.shape}")

generation_actual            (3836309, 5)
load_actual                  (182005, 5)
load_forecast_da             (182245, 5)
outages                      (193994, 14)
transfer_capacity_da         (273980, 5)
wind_solar_forecast_da       (362210, 5)
wind_solar_forecast_intraday (362970, 5)
